In [1]:
from datetime import datetime
from pprint import pprint
from pathlib import Path
import sys

# Find repo root
repo_root = Path('.').resolve()
for _ in range(5):
    if (repo_root / 'shared_utils').is_dir():
        break
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

from umbra import umbra_v2
from shared_utils import s3utils
from shared_utils import cog_utils

Repo root: /home/jovyan/disasters-product-algorithms


In [2]:
###### CONFIG #######
bucket = "csda-data-vendor-umbra"
prefix = "disasters"
upload_bucket = "s3://nasa-disasters/test_uploads"

# Target CRS for the output COG.
# None preserves the source projection (fastest, no warp).
# Uncomment the EPSG:3857 line if you plan to push the COG through
# veda-data-airflow's build_stac, which trips on the EPSG:4326 ensemble.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"


In [3]:
tifs = umbra_v2.retrieve_umbra_resources(datetime(2026, 1, 28, 16), bucket, prefix)

pprint(tifs)

['s3://csda-data-vendor-umbra/disasters/170214af-0869-46c0-8df5-e2fe30723d85/2026-01-28-17-38-00_UMBRA-10/2026-01-28-17-38-00_UMBRA-10_CSI.tif',
 's3://csda-data-vendor-umbra/disasters/170214af-0869-46c0-8df5-e2fe30723d85/2026-01-28-17-38-00_UMBRA-10/2026-01-28-17-38-00_UMBRA-10_GEC.tif']


In [4]:
sig = umbra_v2.sigmaCalib(tifs)

GEC file not found, downloading from s3
Generating Sigma Naught

	* Opening GEC File
65535 1
{'COLLECT_ID': '2eb549d8-60f3-4500-a7ee-2e7525316921', 'DN_TO_BETA': '0.0006118570622538146', 'DN_TO_GAMMA': '0.0005512872602672962', 'DN_TO_RCS': '0.00019257371166149766', 'DN_TO_SIGMA': '0.00048253264059943237', 'PROCESSOR': '4.32.0', 'AREA_OR_POINT': 'Area'}
0.00048253264059943237
<class 'str'>
30.0 -66.32946607530499
Generation completed, file saved to /tmp/s3_temp/202601_Umbra-10_sigma02026-01-28T17:38:00Z.tif


In [5]:
beta = umbra_v2.betaCalib(tifs)

GEC file found, proceeding
Generating Beta Naught

	* Opening GEC File
65535 1
{'COLLECT_ID': '2eb549d8-60f3-4500-a7ee-2e7525316921', 'DN_TO_BETA': '0.0006118570622538146', 'DN_TO_GAMMA': '0.0005512872602672962', 'DN_TO_RCS': '0.00019257371166149766', 'DN_TO_SIGMA': '0.00048253264059943237', 'PROCESSOR': '4.32.0', 'AREA_OR_POINT': 'Area'}
0.0006118570622538146
<class 'str'>
32.062465618710846 -64.26700045659415
Generation completed, file saved to /tmp/s3_temp/202601_Umbra-10_beta02026-01-28T17:38:00Z.tif


In [6]:
gamma = umbra_v2.gammaCalib(tifs)

GEC file found, proceeding
Generating Gamma Naught

	* Opening GEC File
65535 1
{'COLLECT_ID': '2eb549d8-60f3-4500-a7ee-2e7525316921', 'DN_TO_BETA': '0.0006118570622538146', 'DN_TO_GAMMA': '0.0005512872602672962', 'DN_TO_RCS': '0.00019257371166149766', 'DN_TO_SIGMA': '0.00048253264059943237', 'PROCESSOR': '4.32.0', 'AREA_OR_POINT': 'Area'}
0.0005512872602672962
<class 'str'>
31.157025204428102 -65.17244087087688
Generation completed, file saved to /tmp/s3_temp/202601_Umbra-10_gamma02026-01-28T17:38:00Z.tif


In [7]:
rcs = umbra_v2.rcsCalib(tifs)

GEC file found, proceeding
Generating RCS Naught

	* Opening GEC File
65535 1
{'COLLECT_ID': '2eb549d8-60f3-4500-a7ee-2e7525316921', 'DN_TO_BETA': '0.0006118570622538146', 'DN_TO_GAMMA': '0.0005512872602672962', 'DN_TO_RCS': '0.00019257371166149766', 'DN_TO_SIGMA': '0.00048253264059943237', 'PROCESSOR': '4.32.0', 'AREA_OR_POINT': 'Area'}
0.00019257371166149766
<class 'str'>
22.02140609664367 -74.30805997866132
Generation completed, file saved to /tmp/s3_temp/202601_Umbra-10_rcs02026-01-28T17:38:00Z.tif


In [8]:
for filepath in [sig, beta, gamma, rcs]:
    cog_filepath = cog_utils.convert_to_cog(filepath, filepath.replace(".tif", "_cog.tif"))
    s3_upload_filepath = s3utils.build_flat_s3_uri(upload_bucket, cog_filepath.split("/")[-1])
    print(s3_upload_filepath)
    s3utils.upload_file_to_s3(cog_filepath, s3_upload_filepath)

  Auto-selected no-data value for float32: -9999.0
  Data type: float32
  No-data value: -9999.0
  Source CRS: EPSG:4326
  Target CRS: EPSG:3857
  Reprojection: Required
  Compression: ZSTD (level 22)
  Overview levels: 5
  Resampling method: bilinear (auto-detected)
  Warping to EPSG:3857...
  Creating output file that is 22106P x 22207L.
Processing /tmp/s3_temp/202601_Umbra-10_sigma02026-01-28T17:38:00Z.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
  Creating COG: 202601_Umbra-10_sigma02026-01-28T17:38:00Z_cog.tif
  ✓ COG created: 202601_Umbra-10_sigma02026-01-28T17:38:00Z_cog.tif
s3://nasa-disasters/test_uploads/202601_Umbra-10_sigma02026-01-28T17:38:00Z_cog.tif
Uploaded: s3://nasa-disasters/test_uploads/202601_Umbra-10_sigma02026-01-28T17:38:00Z_cog.tif
  Auto-selected no-data value for float32: -9999.0
  Data type: float32
  No-data value: -9999.0
  Source CRS: EPSG:4326
  Target CRS: EPSG:3857
  Reprojection: Required
  Compression: ZSTD (level 22)
  Ov

In [9]:
# This cleans out the s3_temp directory currently used for storing the 
# downloaded base geotiffs and produced rasters
s3utils.remove_s3_temp()